# **THU THẬP DỮ LIỆU**

---

**Mục tiêu:** Thu thập dữ liệu từ sàn thương mại điện tử bằng Web Crawling hoặc API

**Thành viên thực hiện:** [Tên - MSSV]

## **1. Import Libraries**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import json
from datetime import datetime
import os

# Selenium (nếu cần)
# from selenium import webdriver
# from selenium.webdriver.common.by import By

import warnings
warnings.filterwarnings('ignore')

## **2. Xác định nguồn dữ liệu**

**Sàn thương mại điện tử:** [Shopee / Lazada / Tiki / ...]

**Phương pháp:** [Web Crawling / API]

**Danh mục sản phẩm:** [Điện thoại / Laptop / Thời trang / ...]

In [ ]:
# Cấu hình
BASE_URL = "https://shopee.vn"  # Thay đổi theo sàn
KEYWORDS = ['điện thoại', 'laptop', 'tai nghe']  # Từ khóa tìm kiếm
TARGET_PRODUCTS = 5000  # Mục tiêu tối thiểu 5000 dòng

# Headers để tránh bị chặn
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json',
    'Referer': BASE_URL
}

## **3. Thiết kế cấu trúc dữ liệu**

Xác định các trường dữ liệu cần thu thập phục vụ bài toán phân tích.

In [ ]:
# Cấu trúc dữ liệu mẫu
data_structure = {
    'product_id': 'ID sản phẩm',
    'product_name': 'Tên sản phẩm',
    'price': 'Giá bán (VNĐ)',
    'original_price': 'Giá gốc (VNĐ)',
    'discount': 'Phần trăm giảm giá (%)',
    'sold': 'Số lượng đã bán',
    'stock': 'Số lượng còn lại',
    'rating': 'Đánh giá trung bình (1-5 sao)',
    'rating_count': 'Số lượt đánh giá',
    'shop_id': 'ID cửa hàng',
    'shop_name': 'Tên cửa hàng',
    'shop_location': 'Địa điểm cửa hàng',
    'category': 'Danh mục sản phẩm',
    'brand': 'Thương hiệu',
    'crawl_date': 'Ngày thu thập'
}

print("Cấu trúc dữ liệu:")
for key, value in data_structure.items():
    print(f"  - {key}: {value}")

## **4. Hàm thu thập dữ liệu**

**Lưu ý quan trọng:**
- Tuân thủ `robots.txt` của website
- Thêm delay giữa các request (1-2 giây)
- Xử lý lỗi và retry khi cần thiết

In [ ]:
def crawl_products(keyword, limit=100):
    """
    Thu thập thông tin sản phẩm theo từ khóa
    
    Args:
        keyword (str): Từ khóa tìm kiếm
        limit (int): Số lượng sản phẩm cần lấy
    
    Returns:
        list: Danh sách sản phẩm
    """
    products = []
    
    # TODO: Implement crawling logic
    # Ví dụ:
    # for page in range(0, limit // 60 + 1):
    #     url = f"{BASE_URL}/api/v4/search/search_items"
    #     params = {'keyword': keyword, 'limit': 60, 'offset': page * 60}
    #     response = requests.get(url, headers=HEADERS, params=params)
    #     
    #     if response.status_code == 200:
    #         data = response.json()
    #         for item in data.get('items', []):
    #             product = parse_product(item)
    #             products.append(product)
    #     
    #     time.sleep(1)  # Rate limiting
    
    print(f"Đã thu thập {len(products)} sản phẩm cho từ khóa '{keyword}'")
    return products


def parse_product(raw_data):
    """
    Parse dữ liệu thô thành cấu trúc mong muốn
    """
    return {
        'product_id': raw_data.get('itemid'),
        'product_name': raw_data.get('name'),
        'price': raw_data.get('price', 0) / 100000,
        'original_price': raw_data.get('price_before_discount', 0) / 100000,
        'discount': raw_data.get('raw_discount', 0),
        'sold': raw_data.get('sold'),
        'stock': raw_data.get('stock'),
        'rating': raw_data.get('item_rating', {}).get('rating_star'),
        'rating_count': raw_data.get('item_rating', {}).get('rating_count', [0])[0],
        'shop_id': raw_data.get('shopid'),
        'shop_name': raw_data.get('shop_name'),
        'shop_location': raw_data.get('shop_location'),
        'category': raw_data.get('catid'),
        'brand': raw_data.get('brand'),
        'crawl_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

## **5. Thực hiện thu thập dữ liệu**

In [ ]:
# Thu thập dữ liệu cho từng từ khóa
all_products = []

for keyword in KEYWORDS:
    print(f"\n{'='*60}")
    print(f"Đang crawl từ khóa: {keyword}")
    print(f"{'='*60}")
    
    products = crawl_products(keyword, limit=2000)
    all_products.extend(products)
    
    # Delay giữa các từ khóa
    time.sleep(2)

print(f"\n✓ Hoàn thành! Tổng số sản phẩm: {len(all_products)}")

## **6. Chuyển đổi sang DataFrame**

In [ ]:
# Tạo DataFrame
df = pd.DataFrame(all_products)

# Kiểm tra kích thước
print(f"Kích thước dữ liệu: {df.shape}")
print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")

# Hiển thị mẫu
df.head()

## **7. Kiểm tra chất lượng dữ liệu ban đầu**

In [ ]:
# Thông tin tổng quan
print("Thông tin dữ liệu:")
df.info()

print("\n" + "="*60)
print("Thống kê mô tả:")
df.describe()

In [ ]:
# Kiểm tra giá trị thiếu
print("Giá trị thiếu (Missing Values):")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Số lượng': missing,
    'Tỷ lệ (%)': missing_pct
})
missing_df[missing_df['Số lượng'] > 0].sort_values('Số lượng', ascending=False)

In [ ]:
# Kiểm tra duplicate
duplicates = df.duplicated(subset=['product_id']).sum()
print(f"Số sản phẩm trùng lặp: {duplicates}")

if duplicates > 0:
    print("\nXử lý: Loại bỏ các bản ghi trùng lặp")
    df = df.drop_duplicates(subset=['product_id'], keep='first')
    print(f"Kích thước sau khi xử lý: {df.shape}")

## **8. Lưu dữ liệu thô**

In [ ]:
# Tạo thư mục nếu chưa có
os.makedirs('../data/raw', exist_ok=True)

# Lưu file CSV
output_file = '../data/raw/products_raw.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Đã lưu dữ liệu vào: {output_file}")
print(f"✓ Tổng số dòng: {len(df):,}")

# Lưu thêm file JSON (backup)
json_file = '../data/raw/products_raw.json'
df.to_json(json_file, orient='records', force_ascii=False, indent=2)
print(f"✓ Đã lưu backup JSON: {json_file}")

## **9. Tổng kết**

**Kết quả thu thập:**
- Tổng số sản phẩm: [Số lượng]
- Số cột dữ liệu: [Số cột]
- Thời gian thu thập: [Thời gian]
- Nguồn dữ liệu: [Tên sàn]

**Bước tiếp theo:**
- Notebook 02: Tiền xử lý dữ liệu (Data Preprocessing)
- Notebook 03: Phân tích khám phá dữ liệu (EDA)